In [21]:
from pyspark.ml.feature import Tokenizer, NGram, HashingTF, MinHashLSH
from pyspark.sql.window import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import size

In [22]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LSH-duplicate-removal") \
    .getOrCreate()

In [23]:
behaviors_path = '/user/ubuntu/dataset/behaviors.tsv'
news_path = '/user/ubuntu/dataset/news.tsv'

In [24]:
news_df = spark.read.csv(
    news_path,
    sep='\t',
    header=False,
    inferSchema=True
)

columns=["news_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]

news_df = news_df.toDF(*columns)

In [25]:
behaviors_df = spark.read.csv(
    behaviors_path,
    sep='\t',
    header=False,
    inferSchema=True
)

columns=["impression_id", "user_id", "time", "history", "impressions"]
behaviors_df = behaviors_df.toDF(*columns)

In [26]:
print("\n DUPLICATE TITLE ANALYSIS")
print("-" * 40)

total_titles = news_df.count()

unique_titles = news_df.select("Title").distinct().count()

exact_duplicate_titles = total_titles - unique_titles

print(f"Total titles: {total_titles:,}")
print(f"Unique titles: {unique_titles:,}")
print(f"Exact duplicate titles: {exact_duplicate_titles:,}")
print(f"Duplicate ratio: {exact_duplicate_titles / total_titles * 100:.2f}%")


 DUPLICATE TITLE ANALYSIS
----------------------------------------


[Stage 7:=========>                                                 (1 + 5) / 6]

Total titles: 101,527
Unique titles: 98,388
Exact duplicate titles: 3,139
Duplicate ratio: 3.09%


In [27]:
print("\n MOST FREQUENTLY DUPLICATED TITLES")
print("-" * 40)

news_df.groupBy("Title") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count")) \
    .show(10, truncate=80)


 MOST FREQUENTLY DUPLICATED TITLES
----------------------------------------
+---------------------------------------------------------------+-----+
|                                                          Title|count|
+---------------------------------------------------------------+-----+
|Powerball Winning Numbers For 10/26/2019 Drawing: $130M Jackpot|   27|
|                                 Evening news briefing from CNN|   18|
|                                                Look of the Day|   18|
|                                                Friday's Scores|   18|
|                                              Photos of the Day|   16|
|                                 Morning news briefing from CNN|   16|
|Powerball Winning Numbers For 10/30/2019 Drawing: $140M Jackpot|   13|
| Powerball Winning Numbers For 10/12/2019 Drawing: $90M Jackpot|   10|
|                                               CNN Business Now|    9|
|                                   Today's weather in St. 

In [28]:
news_lsh_df = news_df \
      .filter(F.col("title").isNotNull()) \
      .withColumn("title_clean", F.lower(F.col("title")))

news_lsh_df = news_lsh_df.withColumn(
    "title_clean",
    F.regexp_replace(F.col("title_clean"), "[^a-zA-Z0-9\\s]", "")
)

In [29]:
tokenizer = Tokenizer(inputCol="title_clean",
                      outputCol = "title_tokens")

news_lsh_df = tokenizer.transform(news_lsh_df)

In [30]:
short_titles_count_two = news_lsh_df.filter(
    size("title_tokens") < 2
).count()


short_titles_count_three = news_lsh_df.filter(
    size("title_tokens") < 3
).count()


short_titles_count_four = news_lsh_df.filter(
    size("title_tokens") < 4
).count()

print(f"Titles with fewer than 2 tokens: {short_titles_count_two}")
print(f"Titles with fewer than 3 tokens: {short_titles_count_three}")
print(f"Titles with fewer than 4 tokens: {short_titles_count_four}")

Titles with fewer than 2 tokens: 2
Titles with fewer than 3 tokens: 93
Titles with fewer than 4 tokens: 377


In [31]:
ngram = NGram(
    n=2,
    inputCol="title_tokens",
    outputCol="title_shingles"
)

news_lsh_df = ngram.transform(news_lsh_df)

In [32]:
hashing_tf = HashingTF(
    inputCol="title_shingles",
    outputCol="title_set",
    binary = True,
    numFeatures = 1 << 18
)

news_lsh_df = hashing_tf.transform(news_lsh_df)

In [33]:
news_lsh_df.select(
    "title",
    "title_tokens",
    "title_shingles",
    "title_set"
).show(5, truncate=False)

+----------------------------------------------------------------------+--------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------+
|title                                                                 |title_tokens                                                                    |title_shingles                                                                                                                               |title_set                                                                                                                         |
+----------------------------------------------------------------------+--------------------------------------------------------------------------

In [34]:
mh = MinHashLSH(
    inputCol = "title_set",
    outputCol="title_minhash",
    numHashTables = 10
)

model = mh.fit(news_lsh_df)

In [36]:
news_lsh_df_filtered = news_lsh_df.filter(F.size(F.col("title_shingles")) > 0)

candidate_pairs = model.approxSimilarityJoin(
    news_lsh_df_filtered,
    news_lsh_df_filtered,
    threshold = 0.2,
    distCol = "JaccardDistance"
).filter("datasetA.news_id != datasetB.news_id") # remove self matches

candidate_pairs.select(
    "datasetA.title", "datasetB.title", "JaccardDistance"
).show(5, truncate=False)

[Stage 44:======================================================> (31 + 1) / 32]

+-----------------------------------------------------------+-----------------------------------------------------------+---------------+
|title                                                      |title                                                      |JaccardDistance|
+-----------------------------------------------------------+-----------------------------------------------------------+---------------+
|Secondary ticket prices fall ahead of World Series Game 7  |Secondary ticket prices fall ahead of World Series Game 7  |0.0            |
|Jamal Adams reportedly wants to be traded to Cowboys       |Jamal Adams reportedly wants to be traded to Cowboys       |0.0            |
|Trump, Apple CEO Reportedly Plan Texas Visit: Reuters      |Trump, Apple CEO Reportedly Plan Texas Visit: Reuters      |0.0            |
|Today's weather in Cincinnati                              |Today's weather in Cincinnati                              |0.0            |
|Watch: Saquon Barkley tosses defe

In [37]:
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf

def jaccard_simmilarity(set1, set2):
  set1, set2 = set(set1), set(set2)
  if not set1 or not set2:
    return 0.0
  return float(len(set1 & set2) / len(set1 | set2))

jaccard_udf = udf(jaccard_simmilarity, FloatType())
verified_pairs = candidate_pairs.withColumn(
    "JaccardSim",
    jaccard_udf(F.col("datasetA.title_shingles"), F.col("datasetB.title_shingles"))
).filter(F.col("JaccardSim") >= 0.8)

In [38]:
duplicates = verified_pairs.select("datasetB.news_id").distinct()

news_cleaned = news_lsh_df.join(
    duplicates,
    news_lsh_df.news_id == duplicates.news_id,
    how="left_anti"
)

orig_count = news_lsh_df.count()
cleaned_count = news_cleaned.count()

print(f"Original articles: {orig_count}")
print(f"After LSH duplicate removal: {cleaned_count}")
print(f"Total number of articles removed: {orig_count - cleaned_count}")

Original articles: 101527
After LSH duplicate removal: 95605
Total number of articles removed: 5922


In [39]:
spark.stop()